# Part 4: Python + SQL Integration 

In [0]:
import sqlite3
import csv

In [0]:
import sqlite3
conn = sqlite3.connect("ecommerce.db")
print("SQLite connection successful!")
conn.close()

SQLite connection successful!


In [0]:
import sqlite3
import csv
import os

DB_PATH = "ecommerce.db"

# Actual location of cleaned CSV files
base_path = "/Workspace/Users/daniharshit0@gmail.com/data/cleaned"

files = {
    "customers": os.path.join(base_path, "customers_clean.csv"),
    "products": os.path.join(base_path, "products_clean.csv"),
    "orders": os.path.join(base_path, "orders_clean.csv"),
    "order_items": os.path.join(base_path, "order_items_clean.csv")
}

# Connect to SQLite
conn = sqlite3.connect(DB_PATH, timeout=30)
cursor = conn.cursor()

cursor.execute("PRAGMA busy_timeout = 30000")

try:

    # 1. REMOVE OLD TABLES

    cursor.execute("DROP TABLE IF EXISTS order_items")
    cursor.execute("DROP TABLE IF EXISTS orders")
    cursor.execute("DROP TABLE IF EXISTS products")
    cursor.execute("DROP TABLE IF EXISTS customers")

    # 2. CREATE CUSTOMERS TABLE

    cursor.execute("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            customer_name TEXT,
            email TEXT,
            registration_date TEXT,
            customer_type TEXT
        )
    """)
    # 3. CREATE PRODUCTS TABLE

    cursor.execute("""
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            product_name TEXT,
            category TEXT,
            subcategory TEXT,
            cost_price REAL
        )
    """)
    # 4. CREATE ORDERS TABLE

    cursor.execute("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER,
            order_date TEXT,
            status TEXT,
            region_code INTEGER
        )
    """)

    # 5. CREATE ORDER ITEMS TABLE

    cursor.execute("""
        CREATE TABLE order_items (
            item_id INTEGER PRIMARY KEY,
            order_id INTEGER,
            product_id INTEGER,
            quantity INTEGER,
            unit_price REAL,
            discount_percent REAL
        )
    """)

    # 6. LOAD CUSTOMERS
    with open(files["customers"], "r", newline="", encoding="utf-8") as file:

        reader = csv.DictReader(file)

        for row in reader:
            cursor.execute("""
                INSERT INTO customers
                (
                    customer_id,
                    customer_name,
                    email,
                    registration_date,
                    customer_type
                )
                VALUES (?, ?, ?, ?, ?)
            """, (
                row["customer_id"],
                row["customer_name"],
                row["email"],
                row["registration_date"],
                row["customer_type"]
            ))

    # 7. LOAD PRODUCTS

    with open(files["products"], "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            cursor.execute("""
                INSERT INTO products
                (
                    product_id,
                    product_name,
                    category,
                    subcategory,
                    cost_price
                )
                VALUES (?, ?, ?, ?, ?)
            """, (
                row["product_id"],
                row["product_name"],
                row["category"],
                row["subcategory"],
                row["cost_price"]
            ))

    # 8. LOAD ORDERS

    with open(files["orders"], "r", newline="", encoding="utf-8") as file:

        reader = csv.DictReader(file)

        for row in reader:
            cursor.execute("""
                INSERT INTO orders
                (
                    order_id,
                    customer_id,
                    order_date,
                    status,
                    region_code
                )
                VALUES (?, ?, ?, ?, ?)
            """, (
                row["order_id"],
                row["customer_id"] if row["customer_id"] != "" else None,
                row["order_date"],
                row["status"],
                row["region_code"]
            ))

    # 9. LOAD ORDER ITEMS

    with open(files["order_items"], "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            cursor.execute("""
                INSERT INTO order_items
                (
                    item_id,
                    order_id,
                    product_id,
                    quantity,
                    unit_price,
                    discount_percent
                )
                VALUES (?, ?, ?, ?, ?, ?)
            """, (
                row["item_id"],
                row["order_id"],
                row["product_id"],
                row["quantity"],
                row["unit_price"],
                row["discount_percent"]
            ))

    # 10. COMMIT

    conn.commit()
    print("Data loaded successfully!")
except Exception as e:
    conn.rollback()
    print("Error while loading data:", e)

finally:
    conn.close()
    print("SQLite connection closed.")

Data loaded successfully!
SQLite connection closed.


### Takes user input for report type (daily/weekly/monthly)

In [0]:
import sqlite3

DB_PATH = "ecommerce.db"

report_type = input("Enter report type (daily/weekly/monthly): ").strip().lower()
start_date = input("Enter start date (YYYY-MM-DD): ").strip()
end_date = input("Enter end date (YYYY-MM-DD): ").strip()

print("\nReport Type:", report_type)
print("Date Range:", start_date, "to", end_date)

Enter report type (daily/weekly/monthly):  monthly

Enter start date (YYYY-MM-DD):  2025-06-31

Enter end date (YYYY-MM-DD):  2025-07-31


Report Type: monthly
Date Range: 2025-06-31 to 2025-07-31


### Calculate total orders, revenue, and unique customers

In [0]:

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
query = """
SELECT COUNT(DISTINCT o.order_id) AS total_orders, ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)), 2) AS total_revenue,
COUNT(DISTINCT o.customer_id) AS unique_customers FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE DATE(o.order_date) BETWEEN DATE(?) AND DATE(?)"""
cursor.execute(query, (start_date, end_date))
result = cursor.fetchone()
total_orders = result[0]
total_revenue = result[1] or 0
unique_customers = result[2]

print("Total Orders:", total_orders)
print("Total Revenue:", total_revenue)
print("Unique Customers:", unique_customers)


===== SUMMARY =====
Total Orders: 196
Total Revenue: 9685971.41
Unique Customers: 178


### Top 3 products 

In [0]:
query = """
SELECT p.product_name, ROUND( SUM( oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)), 2) 
AS product_revenue FROM orders o JOIN order_items oi ON o.order_id = oi.order_id JOIN products p
ON oi.product_id = p.product_id WHERE DATE(o.order_date) BETWEEN DATE(?) AND DATE(?) GROUP BY p.product_id, p.product_name
ORDER BY product_revenue DESC LIMIT 3
"""
cursor.execute(query, (start_date, end_date))
top_products = cursor.fetchall()

for i, product in enumerate(top_products, 1):
    print(i, product[0],"- Revenue:",product[1])

1 Monitor - Revenue: 829718.59
2 Laptop - Revenue: 591499.61
3 Keyboard - Revenue: 527996.83


### Comparison with previous period (% change) 

In [0]:
def percentage_change(current, previous):
    if previous == 0:
        return 0
    return round(((current - previous) / previous) * 100, 2)
orders_change = percentage_change(
    total_orders,
    previous_orders
)
revenue_change = percentage_change(
    total_revenue,
    previous_revenue
)
customers_change = percentage_change(
    unique_customers,
    previo
print("Orders Change:", orders_change, "%")
print("Revenue Change:", revenue_change, "%")
print("Customers Change:", customers_change, "%")


===== PERCENTAGE CHANGE =====
Orders Change: -87.65 %
Revenue Change: -88.01 %
Customers Change: -78.27 %


# EDGE CASE HANDLING 

In [0]:
import sqlite3
DB_PATH = "ecommerce.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
print("SQLite connected successfully!")

SQLite connected successfully!


### Test Case 1: order_items with an invalid order_id.

In [0]:
def test_invalid_order_id():
    query = """
    SELECT
        oi.item_id,
        oi.order_id
    FROM order_items oi
    LEFT JOIN orders o
        ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
    """
    cursor.execute(query)
    invalid_items = cursor.fetchall()
    if invalid_items:
        print("Invalid order_id found!")
        print("Affected order items:")
        for item in invalid_items:
            print("Item ID:", item[0], "| Order ID:", item[1])
    else:
        print("No invalid order_id found.")
        print("All order_items belong to an existing order.")
test_invalid_order_id()

===== TEST 1: INVALID ORDER_ID =====
No invalid order_id found.
All order_items belong to an existing order.


### Test Case 2 — discount_percent > 100

In [0]:
def test_discount_greater_than_100():
    query = """
    SELECT
        item_id,
        product_id,
        discount_percent
    FROM order_items
    WHERE discount_percent > 100
    """

    cursor.execute(query)
    invalid_discounts = cursor.fetchall()

    if invalid_discounts:
        print("Invalid discount found!")

        for row in invalid_discounts:
            print(
                "Item ID:", row[0],
                "| Product ID:", row[1],
                "| Discount:", row[2]
            )
    else:
        print("No discount greater than 100 found.")
        print("All discount values are valid.")


test_discount_greater_than_100()

===== TEST 2: DISCOUNT > 100 =====
No discount greater than 100 found.
All discount values are valid.


### Test Case 3 — quantity = 0

In [0]:
def test_zero_quantity():
    query = """
    SELECT
        item_id,
        order_id,
        product_id,
        quantity
    FROM order_items
    WHERE quantity = 0
    """

    cursor.execute(query)
    zero_quantity_items = cursor.fetchall()

    if zero_quantity_items:
        print("Zero quantity found!")

        for row in zero_quantity_items:
            print(
                "Item ID:", row[0],
                "| Order ID:", row[1],
                "| Product ID:", row[2],
                "| Quantity:", row[3]
            )
    else:
        print("No quantity equal to 0 found.")
        print("All order items have a positive quantity.")


test_zero_quantity()

===== TEST 3: QUANTITY = 0 =====
No quantity equal to 0 found.
All order items have a positive quantity.


### Test Case 4 — order_date is in the future

In [0]:
from datetime import datetime

def test_future_order_date():
    query = """
    SELECT
        order_id,
        customer_id,
        order_date
    FROM orders
    WHERE DATE(order_date) > DATE('now')
    """

    cursor.execute(query)
    future_orders = cursor.fetchall()

    if future_orders:
        print("Future order date found!")

        for row in future_orders:
            print(
                "Order ID:", row[0],
                "| Customer ID:", row[1],
                "| Order Date:", row[2]
            )
    else:
        print("No future order date found.")
        print("All order dates are valid.")


test_future_order_date()

===== TEST 4: FUTURE ORDER DATE =====
No future order date found.
All order dates are valid.
